# 01 — What is actually in a Xenium run?

**Day 1, 10:00–10:45**

### Where we are going
A Xenium output folder is not a count matrix with extra columns. It is four
different kinds of object glued together, and most confusion later comes from not
knowing which one you are holding.

By the end of this notebook you can:

1. name the four data layers a Xenium run produces, and say what resolution each has
2. load the count matrix into `AnnData` and find the spatial coordinates
3. explain why `adata.obsm["spatial"]` is in **microns**, not pixels, and why that matters
4. read the transcript table and the segmentation polygons, and see how they relate
5. say precisely what is different about this object compared to a scRNA-seq `AnnData`

## The four layers

| Layer | File in `outs/` | One row is... | Resolution |
|---|---|---|---|
| **Counts** | `cell_feature_matrix.h5` | a cell x gene count | cell |
| **Cell metadata** | `cells.parquet` | a cell: centroid, area, control counts | cell |
| **Transcripts** | `transcripts.parquet` | one decoded molecule: x, y, z, gene | sub-cellular |
| **Segmentation** | `cell_boundaries.parquet`, `nucleus_boundaries.parquet` | one polygon vertex | sub-cellular |
| **Morphology** | `morphology_focus/*.ome.tif` | a pixel: DAPI + boundary stains | ~0.2 um/px |

Plus `experiment.xenium` (run metadata, pixel size), `gene_panel.json` (the panel),
`metrics_summary.csv` (10x's own run QC) and an `analysis/` folder with 10x's
default clustering — which you should treat as a first look, never as an answer.

**The single most important sentence in this notebook:** the count matrix is a
*derived* product. It exists because somebody drew polygons around nuclei and
assigned transcripts to them. Every layer below it is closer to the measurement.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

## 1. The count matrix

We load a **spatial crop** of the full run — a contiguous rectangle of tissue,
roughly 1.5 x 1.5 mm. Not a random sample of cells: everything in notebooks 04 and
05 depends on having *all* the neighbours of every cell.

In [ ]:
adata = sc.read_h5ad(DATA / "ovarian_subset.h5ad")
adata

Read that printout carefully. Three things to notice:

- `n_obs x n_vars` — thousands of cells, ~5000 features. Compare to a scRNA-seq
  object: ~20,000 genes. **You only measure what is on the panel.**
- `obs` carries columns you have never seen in scRNA-seq: `cell_area`,
  `nucleus_area`, `control_probe_counts`.
- `obsm['spatial']` — an (n_cells, 2) array. This is the whole difference.

### A short detour: what is an object, and what does the dot do?

If you have not used Python much, this is the one piece of syntax worth understanding
before we go further, because everything below uses it.

`adata` is an **object**. Objects are very handy for curating data: instead of five
loose variables that can drift out of sync, everything about this experiment lives in
one place — the counts, the per-cell table, the per-gene table, the coordinates.

You reach inside with a **dot**.

```python
adata.obs        # the per-cell table (a pandas DataFrame)
adata.var        # the per-gene table
adata.X          # the count matrix itself
adata.obsm       # extra per-cell arrays, including "spatial"
```

**Functions can live inside objects too**, and you call them with the same dot. A
function attached to an object is called a *method*:

```python
adata.obs.head()          # show the first rows
adata.obs.describe()      # summary statistics
adata.var_names.tolist()  # convert to a plain list
```

**The parentheses are the difference that catches people.** No parentheses means you
are asking for a piece of *data*; parentheses mean you are asking the object to *do*
something.

```python
adata.obs.shape     # a property: (n_cells, n_columns)  -- no parentheses
adata.obs.head()    # a method: run it and give me the result  -- parentheses
adata.obs.head      # forgetting them prints a description of the function, not data
```

You can chain dots, reading left to right: `adata.obs.head()` means "in `adata`, take
`obs`, then run `head` on it".

In [ ]:
# Try each of these. The dot is the only new syntax here.
print("adata.shape           ->", adata.shape)          # property, no ()
print("adata.n_obs           ->", adata.n_obs)          # property
print("adata.obs.shape       ->", adata.obs.shape)      # property of a property
print("adata.obs.columns[:4] ->", list(adata.obs.columns[:4]))
print()
print("adata.obs.head(3) — a method, so it needs the parentheses:")
display(adata.obs.head(3))

> **Coming from R?** The ideas are the same, the punctuation is not.
>
> | Task | R / Seurat | Python / scanpy |
> |---|---|---|
> | per-cell metadata | `seurat@meta.data` or `seurat[[]]` | `adata.obs` |
> | per-gene metadata | `seurat[["RNA"]][[]]` (meta.features) | `adata.var` |
> | first rows | `head(df)` | `df.head()` |
> | dimensions | `dim(obj)` | `obj.shape` |
> | column of a table | `df$total_counts` | `df["total_counts"]` |
> | number of cells | `ncol(seurat)` | `adata.n_obs` |
> | number of genes | `nrow(seurat)` | `adata.n_vars` |
>
> Three differences worth internalising:
>
> - **`head(df)` versus `df.head()`.** In R most functions are standalone and you pass
>   the data in. In Python the function often belongs to the object, so it comes after
>   the dot. Both styles exist in both languages, but this is the common case.
> - **`$` versus `.`** — in R the dot is just an ordinary character in a name
>   (`data.frame`, `my.var`); it carries no meaning. In Python it always means "look
>   inside this thing".
> - **Cells are rows here.** A Seurat object is genes × cells; `AnnData` is
>   cells × genes. So `adata.shape` reads the opposite way round from `dim(seurat)`,
>   and `adata.obs` has one row per cell.
>
> Also: Python counts from 0, R from 1.

**A trick worth knowing.** Type `adata.` and press **Tab** in a notebook cell — Jupyter
lists everything inside the object. Same for `adata.obs.`. It is the fastest way to
find out what is available without looking anything up.

In [ ]:
# your code here! try to print the head of the obs dataframe that lives inside the adata object.

In [ ]:
print("spatial coordinates, first five cells (microns):")
print(adata.obsm["spatial"][:5])

x, y = adata.obsm["spatial"].T
print(f"\nextent: x {x.min():.0f}-{x.max():.0f} um, y {y.min():.0f}-{y.max():.0f} um")
print(f"area:   {(x.max()-x.min()) * (y.max()-y.min()) / 1e6:.2f} mm^2")
print(f"density: {adata.n_obs / ((x.max()-x.min())*(y.max()-y.min()) / 1e6):,.0f} cells/mm^2")

> **Microns, not pixels.** Xenium reports centroids in physical units. That is a gift:
> a distance of 30 um means the same thing in every dataset from every instrument,
> so you can talk about "cells within one cell diameter" and have it mean something
> biological. Visium and image-derived coordinates are often in pixels and you have
> to carry a scale factor around. Check the units before you compute any distance.

## 2. The panel, and the controls hiding in it

A Prime 5K run does not only probe genes. It also runs several classes of control
codeword. These are *rows in your count matrix* and if you leave them in, they get
normalised, clustered on, and reported as markers.

In [ ]:
if "feature_types" in adata.var.columns:
    print(adata.var["feature_types"].value_counts(), "\n")

print(f"targeted genes:   {int((~adata.var['control']).sum()):>5}")
print(f"control features: {int(adata.var['control'].sum()):>5}\n")
print("examples of control feature names:")
print(list(adata.var_names[adata.var["control"]][:8]))

What each control is for:

- **Negative control probe** — a probe against a sequence not present in the tissue.
  Fires only if probes bind non-specifically. Measures *chemistry* background.
- **Negative control codeword** — a codeword in the decoding scheme that no probe uses.
  Fires only if the decoder makes mistakes. Measures *optical/decoding* background.
- **Unassigned codeword** — decoded to something not in the panel.

Splitting them tells you *which* part of the pipeline is misbehaving. We will use
them properly in notebook 02.

### Exercise 1.1
Which single control feature has the highest total count in this crop? Is that
worrying, or is it what you would expect?

In [ ]:
# your code here

## 3. The transcript table — this is the raw measurement

Below the cell x gene matrix sits a table with one row per decoded molecule. This is
the layer that has no equivalent whatsoever in single-cell RNA-seq. We ship it for a
small 400 x 400 um window because the full table for this run is tens of gigabytes.

In [ ]:
tx = pd.read_parquet(DATA / "transcripts_crop.parquet")
print(f"{len(tx):,} transcripts in a 400 x 400 um window")
tx.head()

In [ ]:
print("columns:", list(tx.columns))
print(f"\nunique genes here: {tx['feature_name'].nunique()}")
print(f"z range: {tx['z_location'].min():.1f} to {tx['z_location'].max():.1f} um  <- it is 3D")

### Where are the transcripts?

Plot a few genes as points. This is the view that makes people understand what
imaging-based spatial transcriptomics actually is.

In [ ]:
# Three genes that mark three different compartments in this section, chosen
# because they are abundant enough to see and spatially distinct from each other.
#   EPCAM  tumour / epithelium
#   DCN    fibroblast / stroma
#   C1QC   macrophages
# Fallbacks are listed in case a gene is missing from your crop.
wanted = [("EPCAM", "CP", "LAPTM4B"),
          ("DCN", "LUM", "COL1A1"),
          ("C1QC", "MS4A6A", "AIF1")]

available = set(tx["feature_name"])
genes = [next((g for g in group if g in available), None) for group in wanted]
genes = [g for g in genes if g]
print("plotting:", genes)

colors = ["#001158", "#F26B43", "#FBAE40"]

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(tx["x_location"], tx["y_location"], s=0.05, c="0.85", linewidths=0, rasterized=True)
for g, c in zip(genes, colors):
    sub = tx[tx["feature_name"] == g]
    ax.scatter(sub["x_location"], sub["y_location"], s=1.5, c=c, label=f"{g} (n={len(sub):,})",
               linewidths=0, rasterized=True)
ax.set_aspect("equal"); ax.invert_yaxis()
ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")
ax.legend(markerscale=6, loc="upper right", framealpha=0.9)
ax.set_title("all transcripts (grey) with three compartments highlighted")
plt.show()

> **Try it yourself — pick your own genes**
>
> The plot above shows three genes. Swap in genes from your own field and re-run.
> The cell below lists what is available and warns you if a name is not on the panel,
> so you cannot break anything.

In [ ]:
# Which genes are on the panel? Search for part of a name.
SEARCH = "COL"                      # <-- CHANGE THIS

hits = sorted(g for g in tx["feature_name"].unique() if SEARCH.upper() in str(g).upper())
print(f"{len(hits)} features matching {SEARCH!r}:")
print(hits[:20])

In [ ]:
# Now plot up to three of them. Names must match exactly (case-sensitive).
MY_GENES = ["EPCAM", "DCN", "C1QC"]        # <-- CHANGE THIS

available = set(tx["feature_name"])
missing = [g for g in MY_GENES if g not in available]
if missing:
    print(f"not on the panel, skipping: {missing}")
MY_GENES = [g for g in MY_GENES if g in available][:3]

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(tx["x_location"], tx["y_location"], s=0.05, c="0.85",
           linewidths=0, rasterized=True)
for g, c in zip(MY_GENES, ["#001158", "#F26B43", "#FBAE40"]):
    sub = tx[tx["feature_name"] == g]
    ax.scatter(sub["x_location"], sub["y_location"], s=1.5, c=c,
               label=f"{g} (n={len(sub):,})", linewidths=0, rasterized=True)
ax.set_aspect("equal"); ax.invert_yaxis()
ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")
ax.legend(markerscale=6, loc="upper right", framealpha=0.9)
plt.show()

for g in MY_GENES:
    print(f"{g:<12} {(tx['feature_name'] == g).sum():>7,} transcripts in this window")

Stop and look at this. You are seeing individual mRNA molecules, in place, in a
tissue section. There is structure visible *before any analysis at all* — that is
the thing worth internalising.

## 4. Segmentation polygons

Now the layer that decides everything downstream: where one cell ends and the next
begins.

In [ ]:
cb = pd.read_parquet(DATA / "cell_boundaries_crop.parquet")
nb = pd.read_parquet(DATA / "nucleus_boundaries_crop.parquet")
print(f"cell polygons:    {cb['cell_id'].nunique():,}  ({len(cb):,} vertices)")
print(f"nucleus polygons: {nb['cell_id'].nunique():,}")
cb.head()

In [ ]:
# zoom right in: a 120 x 120 um patch, with transcripts and boundaries together
x0 = tx["x_location"].min() + 100
y0 = tx["y_location"].min() + 100
W = 120

def in_box(df, xc, yc):
    return df[(df[xc] >= x0) & (df[xc] < x0 + W) & (df[yc] >= y0) & (df[yc] < y0 + W)]

tx_z = in_box(tx, "x_location", "y_location")
ids = set(in_box(cb, "vertex_x", "vertex_y")["cell_id"])

fig, ax = plt.subplots(figsize=(7, 7))
for cid in ids:
    poly = cb[cb["cell_id"] == cid]
    ax.plot(poly["vertex_x"], poly["vertex_y"], lw=0.7, color="#001158")
    npoly = nb[nb["cell_id"] == cid]
    if len(npoly):
        ax.fill(npoly["vertex_x"], npoly["vertex_y"], color="#BCD2FF", alpha=0.55, lw=0)
ax.scatter(tx_z["x_location"], tx_z["y_location"], s=3, c="#F26B43", linewidths=0)
ax.set_xlim(x0, x0 + W); ax.set_ylim(y0, y0 + W)
ax.set_aspect("equal"); ax.invert_yaxis()
ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")
ax.set_title("nuclei (blue fill), cell boundaries (outline), transcripts (orange)")
plt.show()

> **Try it yourself — move the microscope**
>
> Change `ZOOM` to see more or less tissue, and `SHIFT_X` / `SHIFT_Y` to look somewhere
> else. Small windows show individual cells; large ones show tissue structure. Find a
> window where you can see a segmentation problem with your own eyes.

In [ ]:
ZOOM = 120.0        # <-- CHANGE THIS: window size in microns (try 60, 200)
SHIFT_X = 100.0     # <-- and this: move right, in microns
SHIFT_Y = 100.0     # <-- and this: move down, in microns

zx = tx["x_location"].min() + SHIFT_X
zy = tx["y_location"].min() + SHIFT_Y

tx_z = tx[(tx["x_location"].between(zx, zx + ZOOM))
          & (tx["y_location"].between(zy, zy + ZOOM))]
ids = set(cb[(cb["vertex_x"].between(zx, zx + ZOOM))
             & (cb["vertex_y"].between(zy, zy + ZOOM))]["cell_id"])
print(f"{len(tx_z):,} transcripts and {len(ids):,} cells in this window")

fig, ax = plt.subplots(figsize=(7, 7))
for cid in ids:
    poly = cb[cb["cell_id"] == cid]
    ax.plot(poly["vertex_x"], poly["vertex_y"], lw=0.7, color="#001158")
ax.scatter(tx_z["x_location"], tx_z["y_location"], s=3, c="#F26B43", linewidths=0)
ax.set_xlim(zx, zx + ZOOM); ax.set_ylim(zy + ZOOM, zy)
ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"{ZOOM:.0f} x {ZOOM:.0f} um")
plt.show()

### Look at this plot like an anatomist, not a bioinformatician

- Are there orange dots **outside** every polygon? Those transcripts are assigned to
  no cell at all. In a typical Xenium run 10–40% of transcripts are unassigned.
- Are there polygons with a transcript sitting right on the boundary? Which cell did
  it go to? The answer is "whichever polygon the pipeline drew", and that is a coin
  flip for that molecule.
- Are the cell boundaries plausible given the nuclei, or do some look like a nucleus
  with a fixed halo stamped around it?

**This is the origin of almost every spatial transcriptomics artefact.** Hold on to it.

### Exercise 1.2
What fraction of transcripts in this window are assigned to a cell? Xenium encodes
"no cell" as `cell_id == "UNASSIGNED"` (some versions use `-1`). Compute it, and
compare the unassigned fraction between a highly expressed gene and a lowly
expressed one.

In [ ]:
# your code here

## 5. The morphology image

The stain the segmentation was built from. Xenium Prime images DAPI plus boundary
stains (18S, ATP1A1, aCD45, E-cadherin), so you get protein-level context for free.

In [ ]:
img_path = DATA / "morphology_crop.ome.tif"
if img_path.exists():
    import tifffile
    img = tifffile.imread(img_path)
    print("image shape:", img.shape, img.dtype)
    plane = img[0] if img.ndim == 3 else img
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(plane, cmap="gray", vmax=np.percentile(plane, 99.5))
    ax.set_title("morphology (channel 0 — DAPI)")
    ax.axis("off")
    plt.show()
else:
    print("morphology_crop.ome.tif not present — skipping (nothing later depends on it)")

## 6. AnnData or SpatialData?

Two object models, and you will meet both.

**`AnnData`** — the scanpy object. One table (cells x genes) plus annotations.
Spatial coordinates live in `obsm["spatial"]`. Everything you know from scanpy works.
It cannot natively hold the image, the polygons or the transcripts.

**`SpatialData`** — a container that holds *all* the layers together, each with its
own coordinate transform: images, labels, shapes (polygons), points (transcripts) and
tables (AnnData). Crop it and every layer crops consistently.

Rule of thumb for this workshop: **do your statistics in `AnnData`, do your
multi-layer plotting and cropping in `SpatialData`.**

In [ ]:
# Optional — only runs if spatialdata is installed.
try:
    import spatialdata as sd
    print("spatialdata", sd.__version__)
    print("""
On the full run you would build one with:

    import spatialdata_io
    sdata = spatialdata_io.xenium("Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs/")
    sdata.write("ovarian.zarr")          # do this once; it is slow
    sdata = sd.read_zarr("ovarian.zarr") # then load lazily, in seconds

    sdata.images     # morphology
    sdata.labels     # segmentation masks
    sdata.shapes     # cell / nucleus polygons
    sdata.points     # transcripts
    sdata.tables     # the AnnData
""")
except ImportError:
    print("spatialdata not installed — fine, nothing else needs it")

## 7. So what is different from scRNA-seq?

Write your own answers before reading ours.

| | scRNA-seq | Xenium |
|---|---|---|
| Genes measured | whole transcriptome (~20k) | fixed panel (5k here) |
| Counts per cell | thousands | hundreds |
| Cell identity | defined by the droplet | defined by **a drawn polygon** |
| Doublets | two cells, one barcode | two cells, one polygon — *and you can look* |
| Ambient RNA | soup, from lysis | spillover, from neighbours — *spatially structured* |
| Dissociation bias | large, silent | absent |
| Position | destroyed | **the measurement** |
| Tissue architecture | inferred, at best | observed |

The last three rows are why you are here. The middle rows are why notebook 02 exists.

### Exercise 1.3 (discussion, no code)
You have a scRNA-seq atlas of ovarian tumours and this Xenium section from one
patient. Name one question each of them can answer that the other cannot. Be
specific — "spatial context" is not an answer, "does CD8 T-cell exhaustion depend on
distance from the tumour nest edge" is.

---
**Next:** `02_quality_control.ipynb` — where we find out how much of this we can trust.